In [ ]:
%matplotlib inline

In [ ]:
# -*- coding: utf-8 -*-
"""
Performance comparée en 2022 (base 100 au 1er janvier) :
convertibles euro vs actions vs obligations IG euro vs obligations HY euro.
Illustre la destruction de convexite : les convertibles sous-performent TOUTES leurs jambes.

Choix des comparateurs obligataires : la composante obligataire d'une convertible
porte du risque de credit corporate. On la compare donc a l'IG euro ET au HY euro
(et non au souverain Bund) pour integrer l'effet conjoint des taux et des spreads.
L'IG (duration plus longue) montre surtout l'effet taux ; le HY (plus court, plus
risque) montre l'effet spread/credit. L'univers convertible euro se situe entre les deux.

"""
import pandas as pd, numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings; warnings.filterwarnings('ignore')

FICHIER = "DATA_MEMOIRE_V3.xlsx"   # a adapter
NAVY, GREY, GREEN, ORANGE = "#1F3864", "#7F7F7F", "#2E7D32", "#D98C00"

df = pd.read_excel(FICHIER, sheet_name='Macro EUR', header=None)
def serie(col_valeur, col_date):
    v = pd.to_numeric(df.iloc[2:, col_valeur], errors='coerce')
    d = pd.to_datetime(df.iloc[2:, col_date], errors='coerce')
    s = pd.Series(v.values, index=d.values).dropna()
    s = s[~s.index.duplicated()].sort_index()
    return s[s.index.year >= 1999]

stoxx = serie(6, 5)     # actions
conv  = serie(26, 27)   # convertibles euro
hy    = serie(28, 29)   # HY euro
ig    = serie(30, 29)   # IG euro

def fenetre_2022(s):
    m = (s.index >= '2022-01-01') & (s.index <= '2022-12-31')
    return s[m]
se, sc, sh, si = map(fenetre_2022, [stoxx, conv, hy, ig])

fig, ax = plt.subplots(figsize=(12, 6.2))
ax.axhline(100, color='#bbb', lw=1)
# (serie, couleur, label, epaisseur, decalage vertical etiquette en points)
courbes = [(se, GREY,   "Actions (Stoxx 600)", 1.8, +6),
           (si, ORANGE, "Obligations IG euro (ICE BofA)", 1.8, -8),
           (sh, GREEN,  "Obligations HY euro (ICE BofA)", 1.8, 0),
           (sc, NAVY,   "Obligations convertibles euro (ICE BofA)", 2.6, 0)]
last_x = se.index[-1]
for s, c, lbl, lw, dy in courbes:
    ax.plot(s.index, s/s.iloc[0]*100, color=c, lw=lw, label=lbl)
    perf = s.iloc[-1]/s.iloc[0]*100 - 100
    ax.annotate(f"{perf:+.1f}%", xy=(last_x, s.iloc[-1]/s.iloc[0]*100),
                xytext=(10, dy), textcoords='offset points', va='center', ha='left',
                fontsize=9.5, fontweight='bold', color=c,
                annotation_clip=False)   # ne pas rogner l'etiquette
# marge a droite pour que les etiquettes tiennent dans le cadre
ax.set_xlim(se.index[0], last_x + pd.Timedelta(days=14))
ax.set_title("La destruction de convexité en 2022 : les convertibles euro ont sous-performé toutes leurs jambes de protection",
             fontsize=12, fontweight='bold', color=NAVY, loc='left', pad=12)
ax.set_ylabel("Performance, base 100 au 1er janvier 2022", fontsize=10)
ax.legend(loc='lower left', fontsize=9.5, framealpha=0.9)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.grid(axis='y', alpha=0.25)
for sp in ['top', 'right']: ax.spines[sp].set_visible(False)
fig.text(0.125, 0.02, "Source : Bloomberg, Indices ICE BofA et Stoxx 600, base 100 au 01/01/2022.",
         fontsize=8, color='#666', style='italic')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("perf_2022.png", dpi=150, bbox_inches='tight')
plt.show()
print("Graphique de performance 2022 genere.")

In [ ]:
# -*- coding: utf-8 -*-
"""
Performance comparee en 2022 aux Etats-Unis (base 100 au 1er janvier) :
convertibles US vs actions (S&P 500) vs IG US vs HY US.

NUANCE A CONNAITRE : aux US, le S&P 500 (-20,0 %) a chute PLUS que les convertibles
(-18,8 %), car l'indice actions est concentre en mega-caps tech, massacrees en 2022.
Les convertibles US ont suivi les actions a la baisse (~ -19 %) sans amorti de leur
plancher obligataire (IG -14,6 %, HY -11,1 %). La destruction de convexite se lit donc
ici dans le fait que les convertibles se comportent comme des actions, pas comme un
instrument protégé par sa composante obligataire.

"""
import pandas as pd, numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings; warnings.filterwarnings('ignore')

FICHIER = "DATA_MEMOIRE_V3.xlsx"   # a adapter
NAVY, GREY, GREEN, ORANGE = "#1F3864", "#7F7F7F", "#2E7D32", "#D98C00"

us = pd.read_excel(FICHIER, sheet_name='Macro US', header=None)
def serie(col_valeur, col_date):
    v = pd.to_numeric(us.iloc[2:, col_valeur], errors='coerce')
    d = pd.to_datetime(us.iloc[2:, col_date], errors='coerce')
    s = pd.Series(v.values, index=d.values).dropna()
    s = s[~s.index.duplicated()].sort_index()
    return s[s.index.year >= 1999]

spx  = serie(6, 5)      # S&P 500
conv = serie(26, 25)    # convertibles US
hy   = serie(30, 29)    # HY US
ig   = serie(32, 31)    # IG US

def fenetre_2022(s):
    m = (s.index >= '2022-01-01') & (s.index <= '2022-12-31')
    return s[m]
se, sc, sh, si = map(fenetre_2022, [spx, conv, hy, ig])

fig, ax = plt.subplots(figsize=(12, 6.2))
ax.axhline(100, color='#bbb', lw=1)
courbes = [(se, GREY,   "Actions (S&P 500)", 1.8, -8),
           (si, ORANGE, "Obligations IG US (ICE BofA)", 1.8, 0),
           (sh, GREEN,  "Obligations HY US (ICE BofA)", 1.8, 0),
           (sc, NAVY,   "Obligations convertibles US (ICE BofA)", 2.6, +6)]
last_x = se.index[-1]
for s, c, lbl, lw, dy in courbes:
    ax.plot(s.index, s/s.iloc[0]*100, color=c, lw=lw, label=lbl)
    perf = s.iloc[-1]/s.iloc[0]*100 - 100
    ax.annotate(f"{perf:+.1f}%", xy=(last_x, s.iloc[-1]/s.iloc[0]*100),
                xytext=(10, dy), textcoords='offset points', va='center', ha='left',
                fontsize=9.5, fontweight='bold', color=c, annotation_clip=False)
ax.set_xlim(se.index[0], last_x + pd.Timedelta(days=14))
ax.set_title("La destruction de convexité aux États-Unis en 2022 : les convertibles ont suivi les actions à la baisse, sans amorti obligataire",
             fontsize=11.5, fontweight='bold', color=NAVY, loc='left', pad=12)
ax.set_ylabel("Performance, base 100 au 1er janvier 2022", fontsize=10)
ax.legend(loc='lower left', fontsize=9.5, framealpha=0.9)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.grid(axis='y', alpha=0.25)
for sp in ['top', 'right']: ax.spines[sp].set_visible(False)
fig.text(0.125, 0.02, "Source : Bloomberg, Indices ICE BofA et S&P 500, base 100 au 01/01/2022.",
         fontsize=8, color='#666', style='italic')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("perf_2022_us.png", dpi=150, bbox_inches='tight')
plt.show()
print("Graphique de performance US 2022 genere.")

In [ ]:
"""
Graphiques de correlation glissante actions-obligations (zone euro et Etats-Unis)
Utilise la feuille "Data Intro" (historique long : US depuis 1962, euro depuis 1989),
en raccordant les series US anciennes (1962-2008) avec la feuille Macro US (2008-2026).

On approxime le rendement obligataire par l'oppose de la variation de taux :
    rendement_obligataire ~ -d(taux)
car prix ~ -duration * d(taux), et la correlation est invariante par la constante duration.
SIGNE :
  correlation NEGATIVE = obligations montent quand actions baissent => elles COUVRENT (regime protecteur)
  correlation POSITIVE = actions et obligations baissent ensemble  => protection detruite (2022+)
"""
import pandas as pd, numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
import warnings; warnings.filterwarnings('ignore')

FICHIER = "DATA_MEMOIRE_V3.xlsx"
NAVY, GREEN, RED = "#1F3864", "#2E7D32", "#C0392B"

def serie(df, col_valeur, col_date, start=3):
    v = pd.to_numeric(df.iloc[start:, col_valeur], errors='coerce')
    d = pd.to_datetime(df.iloc[start:, col_date], errors='coerce')
    s = pd.Series(v.values, index=d.values).dropna()
    return s[~s.index.duplicated()].sort_index()

def correlation(actions, taux, fenetre):
    d = pd.concat([actions.rename('a'), taux.rename('t')], axis=1, sort=True).dropna()
    return np.log(d['a']).diff().rolling(fenetre).corr(-d['t'].diff()).dropna()

def figure(actions, taux, titre, source, sortie, regimes):
    c6  = correlation(actions, taux, 126)   # ~6 mois
    c12 = correlation(actions, taux, 252)   # ~1 an
    fig, ax = plt.subplots(figsize=(13, 6.2))
    ax.fill_between(c12.index, 0, 1, where=c12 > 0, transform=ax.get_xaxis_transform(), color=GREEN, alpha=0.08)
    ax.fill_between(c12.index, 0, 1, where=c12 <= 0, transform=ax.get_xaxis_transform(), color=RED, alpha=0.08)
    ax.axhline(0, color='#888', lw=0.9)
    ax.plot(c6.index, c6.values, color=NAVY, lw=0.6, alpha=0.30, label="Fenetre glissante 6 mois")
    ax.plot(c12.index, c12.values, color=NAVY, lw=1.9, label="Fenetre glissante 1 an")
    for dt, txt in regimes:
        dt = pd.Timestamp(dt)
        ax.axvline(dt, color=RED, lw=0.9, ls='--', alpha=0.6)
        ax.annotate(txt, xy=(dt, 0.78), rotation=90, va='top', ha='right', fontsize=8, color=RED, fontstyle='italic')
    ax.set_ylim(-0.88, 0.88)
    ax.set_title(titre, fontsize=13, fontweight='bold', color=NAVY, loc='left', pad=12)
    ax.set_ylabel("Correlation actions / obligations", fontsize=10)
    bandes = [Patch(facecolor=GREEN, alpha=0.2, label="Correlation positive (les obligations ne protegent plus)"),
              Patch(facecolor=RED, alpha=0.2, label="Correlation negative (les obligations couvrent les actions)")]
    ax.add_artist(ax.legend(handles=bandes, loc='upper left', fontsize=8.3, framealpha=0.9))
    ax.legend(loc='lower left', fontsize=9, framealpha=0.9)
    ax.xaxis.set_major_locator(mdates.YearLocator(5))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.grid(axis='y', alpha=0.25)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    fig.text(0.125, 0.02, source, fontsize=8, color='#666', style='italic')
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.savefig("perf_2022.png", dpi=150, bbox_inches='tight')
    plt.show()

# ----- Chargement -----
di  = pd.read_excel(FICHIER, sheet_name='Data Intro', header=None)
mus = pd.read_excel(FICHIER, sheet_name='Macro US',  header=None)

# Euro depuis 1989 (Data Intro) : SXXP col5/date4, GDBR10 col7/date6
stoxx = serie(di, 5, 4)
bund  = serie(di, 7, 6)

# US : raccord 1962-2008 (Data Intro: SPX col1/date0, UST col3/date2) + 2008-2026 (Macro US: SPX col6/date5, UST col14/date13)
spx_old = serie(di, 1, 0);  ust_old = serie(di, 3, 2)
spx_new = serie(mus, 6, 5, start=2); ust_new = serie(mus, 14, 13, start=2)
spx = pd.concat([spx_old[spx_old.index < '2008-01-01'], spx_new[spx_new.index >= '2008-01-01']]).sort_index()
ust = pd.concat([ust_old[ust_old.index < '2008-01-01'], ust_new[ust_new.index >= '2008-01-01']]).sort_index()
spx = spx[~spx.index.duplicated()]; ust = ust[~ust.index.duplicated()]

figure(spx, ust,
    "Correlation glissante actions-obligations aux Etats-Unis depuis 1962 (S&P 500 / Treasury 10 ans)",
    "Source : Bloomberg",
    "corr_us.png",
    [("1970-01-01","Stagflation"), ("1994-01-01","Great Bond Massacre"),
     ("2000-06-01","Bascule vers correlation negative"), ("2022-01-01","Retour correlation positive")])

figure(stoxx, bund,
    "Correlation glissante actions-obligations en zone euro depuis 1989 (Stoxx 600 / Bund 10 ans)",
    "Source : Bloomberg.",
    "corr_euro.png",
    [("2000-06-01","Bascule vers correlation negative"), ("2022-01-01","Retour correlation positive")])

In [ ]:
# -*- coding: utf-8 -*-
"""
Diagramme de payoff d'une obligation convertible selon le regime (schema theorique, chapitre I.1).
Quatre zones : distressed (decrochement sous le plancher, risque de defaut),
obligataire (bond-like), equilibree (balanced, convexite maximale, delta 40-60%),
action (equity-like). Reprend la structure des schemas de gerants specialises,
mais avec les bornes propres au memoire (delta 40-60%) et un rendu fait maison.

Schema purement illustratif (aucune donnee).
"""
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

N = 100.0; Cr = 1.0; K = N/Cr; bond_floor = 88.0
S = np.linspace(20, 170, 500)
parity = S * Cr
beta = 0.06
smooth_env = (1/beta)*np.log(np.exp(beta*bond_floor) + np.exp(beta*parity))
option_premium = 12*np.exp(-((S - K)/40)**2)
cb_base = smooth_env + option_premium
# Decrochement distressed (sigmoide lisse) : la convertible glisse sous le plancher quand S tres bas
distress_factor = 1/(1 + np.exp((S - 42)/6))
distress = np.clip(distress_factor*(bond_floor - 0.55*S - 2), 0, None)
cb_value = cb_base - distress

NAVY, ORANGE, GREEN, RED, GREY = "#1F3864", "#D98C00", "#2E7D32", "#C0392B", "#888"
fig, ax = plt.subplots(figsize=(12, 7.2))
z = [20, 48, 72, 128, 150, 170]
ax.axvspan(z[0], z[1], alpha=0.07, color=RED)
ax.axvspan(z[1], z[2], alpha=0.05, color=ORANGE)
ax.axvspan(z[2], z[3], alpha=0.07, color='#4472C4')
ax.axvspan(z[3], z[5], alpha=0.05, color=GREEN)
ax.axhline(bond_floor, color=ORANGE, lw=2, ls='--', label="Plancher obligataire (bond floor)")
ax.plot(S, parity, color=GREEN, lw=2, ls='--', label="Parité (valeur de conversion)")
ax.plot(S, cb_value, color=NAVY, lw=3, label="Valeur de l'obligation convertible")
ytop = 176
ax.text(34, ytop, "Distressed", ha='center', fontsize=10.5, color=RED, fontweight='bold')
ax.text(60, ytop, "Obligataire\n(bond-like)", ha='center', fontsize=10.5, color=ORANGE, fontweight='bold')
ax.text(100, ytop, "Équilibrée\n(balanced)", ha='center', fontsize=10.5, color='#2E5496', fontweight='bold')
ax.text(159, ytop, "Action\n(equity-like)", ha='center', fontsize=10.5, color=GREEN, fontweight='bold')
ax.text(34, 150, "• Risque de défaut\n• Décrochement\n  sous le plancher", ha='center', fontsize=8, color=RED, va='top')
ax.text(60, 140, "• Delta faible\n• Focus rendement\n  à maturité (YTM)", ha='center', fontsize=8, color='#9C6500', va='top')
ax.text(110, 80, "• Delta 40 à 60 %\n• Convexité et asymétrie\n  maximales\n• Portage (coupon − dividende)", ha='center', fontsize=8, color='#2E5496', va='top')
ax.text(150, 118, "• Delta élevé\n• Suit l'action\n• Faible prime", ha='center', fontsize=8, color=GREEN, va='top')
ia = np.argmin(abs(S - 90))
ax.annotate("", xy=(95, cb_value[ia]), xytext=(95, smooth_env[ia]), arrowprops=dict(arrowstyle='<->', color=GREY, lw=1.3))
ax.text(94, (cb_value[ia]+smooth_env[ia])/2.11+1, "prime\n(valeur option)", fontsize=8.5, color=GREY, va='center', ha='right')
ax.axvline(K, color=GREY, lw=0.8, ls=':')
ax.text(K+1.5, 30, "Prix de conversion K", fontsize=8.5, color=GREY, rotation=90, va='bottom')
ax.set_xlabel("Cours de l'action sous-jacente (S)", fontsize=11)
ax.set_ylabel("Valeur de l'obligation convertible", fontsize=11)
ax.set_title("Profil de valorisation d'une obligation convertible selon le régime", fontsize=13.5, fontweight='bold', color=NAVY, loc='left', pad=14)
ax.legend(loc='lower right', fontsize=9.5, framealpha=0.95)
ax.set_xlim(20, 170); ax.set_ylim(25, 185)
ax.grid(alpha=0.18)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
fig.text(0.125, 0.012, "Schéma théorique, à visée illustrative.", fontsize=8, color='#666', style='italic')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("payoff_convexe.png", dpi=150, bbox_inches='tight')
plt.show()
print("Diagramme de payoff (4 zones) genere.")

In [ ]:
# -*- coding: utf-8 -*-
"""
Delta et gamma d'une obligation convertible en fonction du cours de l'action (chapitre I.3).
Delta : courbe en S de 0 (obligataire) a 1 (action). Gamma : cloche, maximale dans la zone
equilibree, ce qui illustre l'hypothese H1 (convexite maximale a delta intermediaire).

Parametres a maturite courte / vol faible : le delta est raide et le pic du gamma se situe
juste sous le strike (a S ~ 98), un decalage reel dans le modele de Black-Scholes du a la
loi log-normale du cours (le gamma est maximal quand d1 = 0, soit un peu sous le strike).

"""
import numpy as np
from scipy.stats import norm
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

K, r, sigma, T = 100.0, 0.02, 0.16, 0.3   # maturite courte / vol faible
S = np.linspace(40, 170, 600)
d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
delta = norm.cdf(d1)
gamma = norm.pdf(d1) / (S*sigma*np.sqrt(T))
gamma_scaled = gamma / gamma.max()          # echelle arbitraire (pour tenir sur la meme figure que le delta)
peak = S[np.argmax(gamma)]                   # position du pic du gamma (~98, juste sous le strike)

NAVY, RED, GREY = "#1F3864", "#C0392B", "#888"
fig, ax1 = plt.subplots(figsize=(11, 6.6))

# Delta (axe de gauche)
ax1.plot(S, delta, color=NAVY, lw=3, label="Delta (échelle de gauche)")
ax1.set_xlabel("Cours de l'action sous-jacente (S)", fontsize=11)
ax1.set_ylabel("Delta", fontsize=11, color=NAVY)
ax1.tick_params(axis='y', labelcolor=NAVY)
ax1.set_ylim(0, 1.05)

# Gamma (axe de droite)
ax2 = ax1.twinx()
ax2.plot(S, gamma_scaled, color=RED, lw=2.5, ls='--', label="Gamma (échelle de droite, unités arbitraires)")
ax2.set_ylabel("Gamma (échelle arbitraire)", fontsize=11, color=RED)
ax2.tick_params(axis='y', labelcolor=RED)
ax2.set_ylim(0, 1.12)

# Strike et pic du gamma annote
ax1.axvline(K, color=GREY, lw=0.9, ls=':')
ax1.text(K+1.5, 0.05, "Strike K", fontsize=9, color=GREY, rotation=90, va='bottom')
ax2.plot(peak, 1.0, 'o', color=RED, ms=6)
ax2.annotate(f"Pic du gamma à S ≈ {peak:.0f},\nlégèrement sous le strike\n(effet de la loi log-normale)",
             xy=(peak, 1.0), xytext=(62, 1.02), fontsize=8.3, color=RED, ha='center',
             arrowprops=dict(arrowstyle='->', color=RED, alpha=0.6))

# Zone equilibree et annotations de regime
ax1.axvspan(85, 115, alpha=0.06, color='#4472C4')
ax1.text(100, 0.2, "Zone équilibrée\n(gamma maximal)", ha='center', fontsize=9.5, color='#2E5496', fontweight='bold')
ax1.annotate("Delta → 0\n(obligataire)", xy=(66, delta[np.argmin(abs(S-66))]), xytext=(60, 0.42),
             fontsize=8.5, color=NAVY, ha='center', arrowprops=dict(arrowstyle='->', color=NAVY, alpha=0.5))
ax1.annotate("Delta → 1\n(action)", xy=(140, delta[np.argmin(abs(S-140))]), xytext=(140, 0.72),
             fontsize=8.5, color=NAVY, ha='center', arrowprops=dict(arrowstyle='->', color=NAVY, alpha=0.5))

ax1.set_title("Delta et gamma d'une obligation convertible en fonction du cours de l'action",
              fontsize=13, fontweight='bold', color=NAVY, loc='left', pad=12)
ax1.set_xlim(40, 170)
ax1.grid(alpha=0.18)
ax1.spines['top'].set_visible(False)
ax2.spines['top'].set_visible(False)

# Legende combinee (les deux axes)
l1, lab1 = ax1.get_legend_handles_labels()
l2, lab2 = ax2.get_legend_handles_labels()
ax1.legend(l1+l2, lab1+lab2, loc='center right', fontsize=9, framealpha=0.95)

fig.text(0.125, 0.012, "Schéma théorique (modèle de Black-Scholes, maturité courte), à visée illustrative.",
         fontsize=8, color='#666', style='italic')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.show()

In [ ]:
# -*- coding: utf-8 -*-
"""
Le plancher obligataire (bond floor) en fonction des taux (chapitre I.3).
Illustre la convexite obligataire : la courbe prix-taux reste au-dessus de sa tangente
(l'approximation par la duration seule). L'ecart courbe/tangente est la convexite obligataire.
"""
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

N, coupon, T = 100.0, 2.0, 5
r = np.linspace(0.0, 0.10, 400)
def bond_price(rate):
    t = np.arange(1, T+1)
    return sum(coupon/(1+rate)**t) + N/(1+rate)**T
bf = np.array([bond_price(x) for x in r])
r0 = 0.03; P0 = bond_price(r0); h = 1e-5
dP = (bond_price(r0+h) - bond_price(r0-h)) / (2*h)
tangente = P0 + dP*(r - r0)

NAVY, ORANGE, GREY = "#1F3864", "#D98C00", "#888"
fig, ax = plt.subplots(figsize=(11, 6.6))
ax.plot(r*100, bf, color=NAVY, lw=3, label="Valeur du plancher obligataire (bond floor)")
ax.plot(r*100, tangente, color=ORANGE, lw=1.8, ls='--', label="Approximation par la duration seule (tangente)")
ax.axvline(r0*100, color=GREY, lw=0.8, ls=':'); ax.plot(r0*100, P0, 'o', color=NAVY, ms=7)
ax.text(r0*100+0.15, P0+3, "taux courant", fontsize=9, color=GREY)
ax.annotate("La courbe réelle est au-dessus\nde la tangente : c'est la\nconvexité obligataire",
            xy=(8.5, bond_price(0.085)), xytext=(6.3, 88), fontsize=9, color=NAVY,
            arrowprops=dict(arrowstyle='->', color=NAVY, alpha=0.6), ha='left')
ax.set_xlabel("Taux d'intérêt (%)", fontsize=11)
ax.set_ylabel("Valeur du plancher obligataire", fontsize=11)
ax.set_title("Le plancher obligataire en fonction des taux : la convexité obligataire", fontsize=13, fontweight='bold', color=NAVY, loc='left', pad=12)
ax.legend(loc='upper right', fontsize=9.5, framealpha=0.95)
ax.set_xlim(0, 10); ax.grid(alpha=0.2)
for s in ['top', 'right']: ax.spines[s].set_visible(False)
fig.text(0.125, 0.012, "Schéma théorique, à visée illustrative.", fontsize=8, color='#666', style='italic')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("bond_floor_taux.png", dpi=150, bbox_inches='tight')
plt.show()
print("Graphique bond floor-taux genere.")

In [ ]:
# -*- coding: utf-8 -*-
"""
Effet de la volatilite implicite sur le delta et le gamma (chapitre I.3, dynamique des grecques).
Double panneau : delta (gauche) et gamma (droite), plusieurs courbes pour differentes volatilites.
Message : une IV faible -> delta raide, gamma haut et etroit ; une IV forte -> delta etale, gamma aplati.
"""
import numpy as np
from scipy.stats import norm
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

K, r = 100.0, 0.02
S = np.linspace(1, 200, 700)   # 0 a 200 (demarre a 1 pour eviter log(0))
def greeks(sigma, T):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    return norm.cdf(d1), norm.pdf(d1)/(S*sigma*np.sqrt(T))

NAVY = "#1F3864"
COLORS = ["#1F77B4", "#E8000B", "#2CA02C", "#9467BD"]   # bleu, rouge, vert, violet (bien distincts)

vols = [0.15, 0.25, 0.35, 0.50]; T_fix = 1.0
cols = COLORS
fig, (axd, axg) = plt.subplots(1, 2, figsize=(13.5, 5.6))
for sig, c in zip(vols, cols):
    d, g = greeks(sig, T_fix)
    axd.plot(S, d, color=c, lw=2.4, label=f"IV = {int(sig*100)} %")
    axg.plot(S, g, color=c, lw=2.4, label=f"IV = {int(sig*100)} %")
for ax in (axd, axg):
    ax.axvline(K, color='#888', lw=0.8, ls=':'); ax.set_xlim(0, 200); ax.grid(alpha=0.2)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    ax.set_xlabel("Cours de l'action (S)", fontsize=10)
axd.set_ylabel("Delta", fontsize=11); axd.set_title("Delta selon la volatilité implicite", fontsize=12, fontweight='bold', color=NAVY, loc='left')
axg.set_ylabel("Gamma", fontsize=11); axg.set_title("Gamma selon la volatilité implicite", fontsize=12, fontweight='bold', color=NAVY, loc='left')
axd.text(K+1.5, 0.05, "Strike K", fontsize=8, color='#888', rotation=90, va='bottom')
axd.legend(fontsize=9, framealpha=0.95, title="Maturité fixée à 1 an"); axg.legend(fontsize=9, framealpha=0.95)
fig.suptitle("Effet de la volatilité implicite sur le delta et le gamma", fontsize=14, fontweight='bold', color=NAVY, x=0.09, ha='left', y=0.99)
fig.text(0.09, 0.01, "Schéma théorique (modèle de Black-Scholes), à visée illustrative.", fontsize=8, color='#666', style='italic')
plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.savefig("greeks_vol.png", dpi=150, bbox_inches='tight')
plt.show()
print("Graphique volatilite genere.")

In [ ]:
# -*- coding: utf-8 -*-
"""
Effet de la maturite sur le delta et le gamma (chapitre I.3, dynamique des grecques).
Double panneau : delta (gauche) et gamma (droite), plusieurs courbes pour differentes maturites.
Message : maturite courte -> delta raide, gamma haut et concentre pres du strike ;
maturite longue -> delta etale, gamma aplati et disperse.
"""
import numpy as np
from scipy.stats import norm
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

K, r = 100.0, 0.02
S = np.linspace(1, 200, 700)   # 0 a 200 (demarre a 1 pour eviter log(0))
def greeks(sigma, T):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    return norm.cdf(d1), norm.pdf(d1)/(S*sigma*np.sqrt(T))

NAVY = "#1F3864"
COLORS = ["#1F77B4", "#E8000B", "#2CA02C", "#9467BD"]   # bleu, rouge, vert, violet (bien distincts)

mats = [1/12, 0.5, 1.0, 3.0]; sig_fix = 0.25
cols = COLORS
fig, (axd, axg) = plt.subplots(1, 2, figsize=(13.5, 5.6))
for T, c in zip(mats, cols):
    d, g = greeks(sig_fix, T)
    lbl = f"T = {T:g} an" + ("s" if T > 1 else "")
    axd.plot(S, d, color=c, lw=2.4, label=lbl)
    axg.plot(S, g, color=c, lw=2.4, label=lbl)
for ax in (axd, axg):
    ax.axvline(K, color='#888', lw=0.8, ls=':'); ax.set_xlim(0, 200); ax.grid(alpha=0.2)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    ax.set_xlabel("Cours de l'action (S)", fontsize=10)
axd.set_ylabel("Delta", fontsize=11); axd.set_title("Delta selon la maturité", fontsize=12, fontweight='bold', color=NAVY, loc='left')
axg.set_ylabel("Gamma", fontsize=11); axg.set_title("Gamma selon la maturité", fontsize=12, fontweight='bold', color=NAVY, loc='left')
axd.text(K+1.5, 0.05, "Strike K", fontsize=8, color='#888', rotation=90, va='bottom')
axd.legend(fontsize=9, framealpha=0.95, title="Volatilité fixée à 25 %"); axg.legend(fontsize=9, framealpha=0.95)
fig.suptitle("Effet de la maturité sur le delta et le gamma", fontsize=14, fontweight='bold', color=NAVY, x=0.09, ha='left', y=0.99)
fig.text(0.09, 0.01, "Schéma théorique (modèle de Black-Scholes), à visée illustrative.", fontsize=8, color='#666', style='italic')
plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.savefig("greeks_maturity.png", dpi=150, bbox_inches='tight')
plt.show()
print("Graphique maturite genere.")

In [ ]:
# =============================================================================
# FIGURES DE REGIME DE CORRELATION ACTIONS-OBLIGATIONS
# Remplace la cellule existante du notebook, et ajoute une figure nouvelle.
# Format notebook : chaque bloc separe par "# %%" est une cellule.
# =============================================================================

# %% CELLULE A - Correlation actions-obligations, version corrigee
# -----------------------------------------------------------------------------
# DEUX CORRECTIONS PAR RAPPORT A LA VERSION PRECEDENTE
#
# 1. Le code couleur etait inverse. Le vert etait applique a la correlation
#    positive, qui est le regime OU LES OBLIGATIONS NE PROTEGENT PLUS. Les
#    libelles de legende etaient justes, mais un lecteur qui regarde les bandes
#    avant de lire la legende comprenait l'inverse.
#    Desormais : vert = regime protecteur, rouge = protection detruite.
#
# 2. Le nom de fichier de sortie etait ecrit en dur ("perf_2022.png"). La
#    fonction recevait un parametre "sortie" qu'elle n'utilisait jamais, donc la
#    figure euro ecrasait la figure US, et les deux ecrasaient une troisieme
#    figure sans rapport. Le parametre est maintenant utilise.
#
# RAPPEL DE CONVENTION, a garder identique dans tout le memoire :
#   On approxime le rendement obligataire par l'oppose de la variation de taux,
#   car prix ~ -duration x d(taux). La duration s'elimine dans une correlation.
#   correlation NEGATIVE = les obligations montent quand les actions baissent,
#                          elles couvrent le portefeuille (regime protecteur)
#   correlation POSITIVE = actions et obligations baissent ensemble,
#                          la protection est detruite (2022 et apres)

import warnings

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")

NAVY, VERT, ROUGE = "#1F3864", "#2E7D32", "#C0392B"

# -----------------------------------------------------------------------------
# LOCALISATION DES FICHIERS
# Python cherche les fichiers dans le dossier ou tourne le notebook, pas la ou
# ils sont ranges. Ta liste ci-dessous indique ou chercher. Tes fichiers sont
# repartis dans plusieurs dossiers, d'ou la liste plutot qu'un dossier unique.
# Ajoute ou retire des lignes librement. Le "r" avant les guillemets est
# obligatoire sur Windows, il empeche Python d'interpreter les antislashs.
DOSSIERS = [
    r"C:\Users\steva\Desktop\MÉMOIRE\OLD",       # DATA_MEMOIRE_V3.xlsx
    r"C:\Users\steva\Desktop\MÉMOIRE\Data VF",   # DATA_MACRO_VF.xlsb, parquets
]


def trouver(nom_fichier):
    """Retourne le chemin complet du fichier, ou leve une erreur explicite."""
    from pathlib import Path

    dossiers = [Path(d) for d in DOSSIERS]
    maison = Path.home()
    dossiers += [
        Path.cwd(),
        maison / "Downloads", maison / "Telechargements", maison / "Téléchargements",
        maison / "Desktop", maison / "Bureau", maison / "Documents",
    ]
    for dossier in dossiers:
        candidat = dossier / nom_fichier
        if candidat.exists():
            return candidat

    extension = Path(nom_fichier).suffix
    lignes = ["", "", "FICHIER INTROUVABLE : " + nom_fichier, "", "Dossiers explores :"]
    for dossier in dossiers:
        etat = "existe" if dossier.is_dir() else "n'existe pas"
        lignes.append("   " + str(dossier) + "   (" + etat + ")")
        if dossier.is_dir():
            visibles = sorted(f.name for f in dossier.glob("*" + extension))
            if visibles:
                lignes.append("      fichiers " + extension + " presents : " + ", ".join(visibles))
    lignes += ["", "Corrige un chemin dans DOSSIERS, en haut de cette cellule.",
               "Verifie aussi le nom exact du fichier, majuscules comprises.", ""]
    raise FileNotFoundError("\n".join(lignes))


FICHIER = trouver("DATA_MEMOIRE_V3.xlsx")


def serie(df, col_valeur, col_date, start=3):
    """Extrait une serie datee depuis une feuille Excel sans en-tete."""
    valeurs = pd.to_numeric(df.iloc[start:, col_valeur], errors="coerce")
    dates = pd.to_datetime(df.iloc[start:, col_date], errors="coerce")
    s = pd.Series(valeurs.values, index=dates.values).dropna()
    return s[~s.index.duplicated()].sort_index()


def correlation(actions, taux, fenetre):
    """Correlation glissante entre rendement action et rendement obligataire approxime."""
    d = pd.concat([actions.rename("a"), taux.rename("t")], axis=1, sort=True).dropna()
    rendement_action = np.log(d["a"]).diff()
    rendement_obligataire = -d["t"].diff()
    return rendement_action.rolling(fenetre).corr(rendement_obligataire).dropna()


def figure_correlation(actions, taux, titre, source, sortie, regimes):
    correlation_6_mois = correlation(actions, taux, 126)
    correlation_1_an = correlation(actions, taux, 252)

    fig, ax = plt.subplots(figsize=(13, 6.2))

    # CORRECTION 1 : positif (protection detruite) en rouge, negatif (protecteur) en vert.
    ax.fill_between(correlation_1_an.index, 0, 1, where=correlation_1_an > 0,
                    transform=ax.get_xaxis_transform(), color=ROUGE, alpha=0.08)
    ax.fill_between(correlation_1_an.index, 0, 1, where=correlation_1_an <= 0,
                    transform=ax.get_xaxis_transform(), color=VERT, alpha=0.08)

    ax.axhline(0, color="#888", lw=0.9)
    ax.plot(correlation_6_mois.index, correlation_6_mois.values, color=NAVY, lw=0.6,
            alpha=0.30, label="Fenetre glissante 6 mois")
    ax.plot(correlation_1_an.index, correlation_1_an.values, color=NAVY, lw=1.9,
            label="Fenetre glissante 1 an")

    for date, texte in regimes:
        date = pd.Timestamp(date)
        ax.axvline(date, color=ROUGE, lw=0.9, ls="--", alpha=0.6)
        ax.annotate(texte, xy=(date, 0.78), rotation=90, va="top", ha="right",
                    fontsize=8, color=ROUGE, fontstyle="italic")

    ax.set_ylim(-0.88, 0.88)
    ax.set_title(titre, fontsize=13, fontweight="bold", color=NAVY, loc="left", pad=12)
    ax.set_ylabel("Correlation actions / obligations", fontsize=10)

    bandes = [
        Patch(facecolor=ROUGE, alpha=0.25,
              label="Correlation positive, les obligations ne protegent plus"),
        Patch(facecolor=VERT, alpha=0.25,
              label="Correlation negative, les obligations couvrent les actions"),
    ]
    ax.add_artist(ax.legend(handles=bandes, loc="upper left", fontsize=8.3, framealpha=0.9))
    ax.legend(loc="lower left", fontsize=9, framealpha=0.9)

    ax.xaxis.set_major_locator(mdates.YearLocator(5))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(axis="y", alpha=0.25)
    for cote in ["top", "right"]:
        ax.spines[cote].set_visible(False)
    fig.text(0.125, 0.02, source, fontsize=8, color="#666", style="italic")
    plt.tight_layout(rect=[0, 0.03, 1, 1])

    plt.savefig(sortie, dpi=150, bbox_inches="tight")   # CORRECTION 2
    plt.show()


data_intro = pd.read_excel(FICHIER, sheet_name="Data Intro", header=None)
macro_us = pd.read_excel(FICHIER, sheet_name="Macro US", header=None)

stoxx = serie(data_intro, 5, 4)
bund = serie(data_intro, 7, 6)

spx_ancien = serie(data_intro, 1, 0)
ust_ancien = serie(data_intro, 3, 2)
spx_recent = serie(macro_us, 6, 5, start=2)
ust_recent = serie(macro_us, 14, 13, start=2)
spx = pd.concat([spx_ancien[spx_ancien.index < "2008-01-01"],
                 spx_recent[spx_recent.index >= "2008-01-01"]]).sort_index()
ust = pd.concat([ust_ancien[ust_ancien.index < "2008-01-01"],
                 ust_recent[ust_recent.index >= "2008-01-01"]]).sort_index()
spx = spx[~spx.index.duplicated()]
ust = ust[~ust.index.duplicated()]

figure_correlation(
    spx, ust,
    "Correlation glissante actions-obligations aux Etats-Unis depuis 1962 (S&P 500 / Treasury 10 ans)",
    "Source : Bloomberg", "corr_us.png",
    [("1970-01-01", "Stagflation"), ("1994-01-01", "Great Bond Massacre"),
     ("2000-06-01", "Bascule vers correlation negative"),
     ("2022-01-01", "Retour correlation positive")],
)

figure_correlation(
    stoxx, bund,
    "Correlation glissante actions-obligations en zone euro depuis 1989 (Stoxx 600 / Bund 10 ans)",
    "Source : Bloomberg", "corr_euro.png",
    [("2000-06-01", "Bascule vers correlation negative"),
     ("2022-01-01", "Retour correlation positive")],
)


# %% CELLULE B - Figure nouvelle : ce qui explique le regime de correlation
# -----------------------------------------------------------------------------
# POURQUOI CETTE FIGURE
# La cellule precedente montre QUE le regime bascule. Elle ne dit pas POURQUOI.
# La litterature (Campbell, Pflueger et Viceira ; Cieslak et Pflueger) attribue
# la bascule a la cyclicite de l'inflation, c'est-a-dire au type de choc :
#
#   inflation PROCYCLIQUE, choc de demande
#       l'inflation monte quand l'activite accelere. En recession l'inflation
#       recule, les taux baissent, les obligations montent pendant que les
#       actions baissent. Les obligations couvrent.
#
#   inflation CONTRACYCLIQUE, choc d'offre
#       l'inflation monte alors que l'activite ralentit. Les taux montent en
#       meme temps que les actions baissent. La protection disparait.
#
# On mesure la cyclicite par la correlation glissante entre les variations du
# point mort d'inflation et le rendement des actions. Positive signifie
# procyclique, donc regime protecteur attendu.
#
# Le point mort d'inflation euro commence en 2004, la figure demarre donc en 2005.

macro = pd.read_parquet(trouver("Data_macro.parquet"))

colonnes = ["Stoxx 600", "Bund 10 ans", "Point mort inflation 10 ans euro"]
base = macro[colonnes].dropna()

rendement_action = np.log(base["Stoxx 600"]).diff()
rendement_obligataire = -base["Bund 10 ans"].diff()
variation_point_mort = base["Point mort inflation 10 ans euro"].diff()

regime = rendement_action.rolling(252).corr(rendement_obligataire)
cyclicite = variation_point_mort.rolling(252).corr(rendement_action)

fig, (haut, bas) = plt.subplots(2, 1, figsize=(13, 8.5), sharex=True,
                                gridspec_kw={"height_ratios": [1, 1], "hspace": 0.12})

# Panneau du haut : le regime observe.
haut.fill_between(regime.index, 0, 1, where=regime > 0,
                  transform=haut.get_xaxis_transform(), color=ROUGE, alpha=0.08)
haut.fill_between(regime.index, 0, 1, where=regime <= 0,
                  transform=haut.get_xaxis_transform(), color=VERT, alpha=0.08)
haut.axhline(0, color="#888", lw=0.9)
haut.plot(regime.index, regime.values, color=NAVY, lw=1.9)
haut.set_ylabel("Correlation actions / obligations", fontsize=10)
haut.set_title(
    "Regime de correlation et cyclicite de l'inflation en zone euro",
    fontsize=13, fontweight="bold", color=NAVY, loc="left", pad=12,
)
haut.legend(handles=[
    Patch(facecolor=ROUGE, alpha=0.25, label="Protection obligataire detruite"),
    Patch(facecolor=VERT, alpha=0.25, label="Regime protecteur"),
], loc="upper left", fontsize=8.3, framealpha=0.9)

# Panneau du bas : le determinant avance par la litterature.
bas.fill_between(cyclicite.index, 0, 1, where=cyclicite <= 0,
                 transform=bas.get_xaxis_transform(), color=ROUGE, alpha=0.08)
bas.fill_between(cyclicite.index, 0, 1, where=cyclicite > 0,
                 transform=bas.get_xaxis_transform(), color=VERT, alpha=0.08)
bas.axhline(0, color="#888", lw=0.9)
bas.plot(cyclicite.index, cyclicite.values, color="#8E44AD", lw=1.9)
bas.set_ylabel("Cyclicite de l'inflation", fontsize=10)
bas.legend(handles=[
    Patch(facecolor=VERT, alpha=0.25, label="Inflation procyclique, choc de demande"),
    Patch(facecolor=ROUGE, alpha=0.25, label="Inflation contracyclique, choc d'offre"),
], loc="upper left", fontsize=8.3, framealpha=0.9)

for axe in (haut, bas):
    axe.grid(axis="y", alpha=0.25)
    for cote in ["top", "right"]:
        axe.spines[cote].set_visible(False)
bas.xaxis.set_major_locator(mdates.YearLocator(2))
bas.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.text(0.125, 0.02,
         "Source : Bloomberg. Cyclicite mesuree par la correlation glissante sur un an entre "
         "variations du point mort d'inflation 10 ans et rendement du Stoxx 600.",
         fontsize=8, color="#666", style="italic")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("regime_et_cyclicite.png", dpi=150, bbox_inches="tight")
plt.show()

lien = pd.DataFrame({"regime": regime, "cyclicite": cyclicite}).dropna()
print("Correlation entre cyclicite de l'inflation et regime : %+.3f"
      % lien["cyclicite"].corr(lien["regime"]))
print("Signe negatif attendu : une inflation contracyclique accompagne la destruction")
print("de la protection obligataire.")
print()
print("Mediane par periode :")
for libelle, debut, fin in [("2005-2007", "2005", "2007"), ("2008-2009", "2008", "2009"),
                            ("2010-2014", "2010", "2014"), ("2015-2019", "2015", "2019"),
                            ("2020-2021", "2020", "2021"), ("2022", "2022", "2022"),
                            ("2023-2026", "2023", "2026")]:
    bloc = lien.loc[debut:fin]
    if bloc.empty:
        continue
    print("  %-10s cyclicite %+.3f   regime %+.3f"
          % (libelle, bloc["cyclicite"].median(), bloc["regime"].median()))